# 04 - Structured output, confidence & tool calling (Om)
Covers brief requirements **9 (structured output)** and **10 (tool / function calling)**; experiments **E4** and **E5**.
Locked: model Qwen2.5-1.5B-Instruct, eval set `data/eval_set.jsonl` (200 q), seed 42, `max_new_tokens=128`, greedy decoding, prompt = the team's best `few_shot` (E2).

**Why this notebook exists:** a free-text answer cannot be checked by code. A fixed JSON schema lets us *verify* every answer
(valid JSON? is the evidence sentence really in the paragraph? does an abstention have an empty answer?) and attach a confidence that comes from token probabilities.

*Colab tip: if the runtime disconnects, just run all cells again - finished rows are saved to `results/` and skipped.*

In [ ]:
# --- Colab setup: Runtime > Change runtime type > T4 GPU, then run this cell ------------------------
import os, sys
REPO = "AppliedGenAIAssignments"
if not os.path.exists(REPO) and not os.path.exists("src/model.py"):
    !git clone https://github.com/OmBoroleGit/AppliedGenAIAssignments.git
if os.path.exists(REPO):
    %cd {REPO}
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())
assert os.path.exists("src/model.py"), "src/model.py missing - upload/push the project files first, then re-run"
!pip install -q -r requirements.txt

In [ ]:
import json, re, os, pandas as pd
from collections import Counter
from src.model import load_model, generate_with_logprobs, set_seed, SEED
from src.schema import AnswerOutput, JSON_SCHEMA, validate_output, answer_json
from src.confidence import calibration_table
from src.runlog import load_eval_set, make_row, make_build_messages, run_cached, COLUMNS, EXTRA
set_seed(SEED)
build_msgs = make_build_messages("few_shot")   # team's best prompt from E2 (results/best_prompt.txt)
tok, model = load_model()
print("model on", model.device)
data = load_eval_set("data/eval_set.jsonl")
print(len(data), "questions;", sum(d["is_answerable"] for d in data), "answerable")

## E4 - Structured output
### 4.1 The schema (design + reason)
| Field | Type | Why it is there |
|---|---|---|
| `answerable` | bool | forces an explicit answer-or-abstain decision (the core of SQuAD v2) |
| `answer` | str | short span; **must be empty** when `answerable=false` (the team's phrase "not in the document" is accepted and stored as empty) |
| `evidence` | str | one sentence copied from the paragraph = checkable proof |
| `confidence` | float 0-1 | optional in the model's reply (team prompts ask for 3 keys); we ADD it from token probabilities (see 4.5) |

In [ ]:
print(json.dumps(JSON_SCHEMA, indent=2))

### 4.2 Validator rules (R1-R7) - demonstration on hand-made replies

In [ ]:
ctx = "Alexander Graham Bell patented the telephone in 1876. He was born in Edinburgh."
tests = {
 "good":              '{"answerable": true, "answer": "1876", "evidence": "Alexander Graham Bell patented the telephone in 1876.", "confidence": 0.8}',
 "good, 3 keys only": '{"answerable": true, "answer": "1876", "evidence": "Alexander Graham Bell patented the telephone in 1876."}',
 "fenced + chatter":  'Sure! ```json\n{"answerable": false, "answer": "", "evidence": ""}\n```',
 "no JSON":           "It was 1876.",
 "invented evidence": '{"answerable": true, "answer": "1877", "evidence": "Bell built it in 1877."}',
 "abstain + answer":  '{"answerable": false, "answer": "1876", "evidence": ""}',
 "team abstain":      '{"answerable": false, "answer": "not in the document", "evidence": ""}',
 "true but refuses":  '{"answerable": true, "answer": "not in the document", "evidence": "x"}',
 "wrong type":        '{"answerable": "yes", "answer": "", "evidence": ""}',
}
for name, raw in tests.items():
    r = validate_output(raw, ctx)
    print(f"{name:18s} ok={r.ok}  {r.errors}")

### 4.3 Run E4 on all 200 questions (greedy, `few_shot` prompt, one retry) - about 10-15 min on a T4

In [ ]:
def run_e4(ex):
    out = answer_json(build_msgs(ex["context"], ex["question"]), ex["context"],
                      generate_with_logprobs, max_retries=1, temperature=0, seed=SEED)
    return make_row(ex, out, prompt_variant="few_shot", decoding="greedy")

df = run_cached("results/E4_structured_output.csv", data, run_e4, key=lambda ex: ex["id"])
df.drop(columns="_key").head()

In [ ]:
n = len(df)
first = df.json_valid_first_attempt.mean(); final = df.json_valid.mean()
print(f"JSON validity, first attempt : {first:.1%}")
print(f"JSON validity, after 1 retry : {final:.1%}   (rescued by retry: {int(round((final-first)*n))} of {n})")
errs = Counter(m for e in df.errors for m in re.findall(r"R\d", e))
print("failed-attempt reasons (R1 no JSON, R2/R3 types, R4 empty fields, R5 invented evidence, R6 abstain+answer, R7 true+refusal):")
print(dict(errs))
print("soft warnings:", Counter(m for w in df.warnings for m in re.findall(r"W\d", w)))
print("confidence source:", dict(df.confidence_source.value_counts()))

### 4.4 Sample validated outputs (paste 3-4 into the report appendix)

In [ ]:
show = pd.concat([df[df.is_answerable].head(4), df[~df.is_answerable].head(2)])
for _, r in show.iterrows():
    print("Q:", r.question, "| gold:", json.loads(r.gold_answers)[:1], "| answerable:", r.is_answerable)
    print("   ->", r.raw_output, "| valid:", r.json_valid, "| conf source:", r.confidence_source, "\n")

### 4.5 Confidence from token probabilities
`confidence = exp(mean log-prob of the decision tokens)` - the tokens after `"answerable":` and inside `"answer":"..."` (see `src/confidence.py`).
Is it *useful*? When the model is more confident, is it more often right? (loose correctness: gold answer found in the prediction, or both abstain)

In [ ]:
def norm(s): return re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", str(s).lower())).strip()
def correct(row):
    gold = json.loads(row["gold_answers"])
    if not row["is_answerable"]: return not row["pred_answerable"]
    return bool(row["pred_answerable"]) and any(norm(g) == norm(row["pred_answer"]) or norm(g) in norm(row["pred_answer"]) for g in gold)
ok = df[df.json_valid].copy()
ok["correct"] = ok.apply(correct, axis=1)
table, ece = calibration_table(ok.confidence.tolist(), ok.correct.tolist(), n_bins=5)
print(pd.DataFrame(table)); print("Expected Calibration Error:", ece)
print("mean confidence when correct :", round(ok[ok.correct].confidence.mean(), 3))
print("mean confidence when wrong   :", round(ok[~ok.correct].confidence.mean(), 3))
print("loose accuracy:", round(ok.correct.mean(), 3))
pd.DataFrame(table).to_csv("results/summary_E4_confidence_calibration.csv", index=False)

## E5 - Tool / function calling: `retrieve_context(question, top_k)`
**Specification** (also in the report): input `{question: str, top_k: int 1-5}`, output `{passages: [{passage_id, title, text, score}]}`.
**Why a tool:** when the user gives a question but no paragraph (or the wrong one), the model cannot answer from nothing - the tool supplies grounded evidence, which is also our RAG option for the adaptation comparison.
**Flow:** model emits a JSON tool call -> Python validates the arguments (pydantic) and runs BM25 -> passages go back as a message -> model writes the final JSON answer -> validator checks the evidence against the *retrieved* passages.

In [ ]:
from datasets import load_dataset
from src import tools
print(json.dumps(tools.TOOL_SPEC, indent=2)[:1400])
sq = load_dataset("rajpurkar/squad_v2", split="validation")
# 20 of the 35 articles in OUR eval set, so demo questions are answerable from the index
titles = []
for d in data:
    t = d.get("title")
    if t and t not in titles: titles.append(t)
    if len(titles) == 20: break
corpus = tools.build_corpus(sq, titles=titles)
print(tools.init_index(corpus), "passages from", len(titles), "articles")
pool = [d for d in data if d.get("title") in set(titles)]
print(len(pool), "eval questions come from these 20 articles (", sum(d["is_answerable"] for d in pool), "answerable )")

### 5.1 Retrieval quality on its own (does BM25 find the gold paragraph?)

In [ ]:
ans_pool = [d for d in pool if d["is_answerable"]]
hit1 = hit3 = 0
for d in ans_pool:
    texts = [p["text"] for p in tools.retrieve_context(d["question"], 3)["passages"]]
    hit1 += bool(texts) and texts[0] == d["context"]
    hit3 += d["context"] in texts
print(f"BM25 hit@1 = {hit1/len(ans_pool):.1%}   hit@3 = {hit3/len(ans_pool):.1%}   (n={len(ans_pool)})")

### 5.2 Tool decision + end-to-end (no paragraph -> model should call the tool; paragraph given -> should not) - about 10 min

In [ ]:
def run_e5(item):
    d, mode = item
    out = tools.answer_with_tools(d["question"], generate_with_logprobs,
                                  paragraph=None if mode == "no_paragraph" else d["context"],
                                  top_k=3, temperature=0, seed=SEED)
    r = make_row(d, out, prompt_variant=f"e5_tools_{mode}", decoding="greedy")
    r.update(mode=mode, tool_called=out["trace"]["tool_called"], forced=out["trace"]["forced_fallback"],
             n_passages=out["trace"]["n_passages"])
    return r

items = [(d, m) for d in pool for m in ("no_paragraph", "with_paragraph")]
d5 = run_cached("results/E5_tool_calling.csv", items, run_e5, key=lambda it: it[0]["id"] + "|" + it[1])
np_ = d5[d5["mode"] == "no_paragraph"]; wp = d5[d5["mode"] == "with_paragraph"]
print(f"no paragraph  : model called the tool itself {np_.tool_called.mean():.1%}, Python had to force the call {np_.forced.mean():.1%}")
print(f"with paragraph: model correctly skipped the tool {(~wp.tool_called).mean():.1%}")
for name, part in (("no_paragraph", np_), ("with_paragraph", wp)):
    v = part[part.json_valid].copy(); v["correct"] = v.apply(correct, axis=1)
    print(f"{name:15s} valid JSON {len(v)}/{len(part)}   loose accuracy {v.correct.mean():.1%}")

### 5.3 Three representative invocations (copy the trace into the report)

In [ ]:
for d in [x for x in pool if x["is_answerable"]][:2] + [x for x in pool if not x["is_answerable"]][:1]:
    out = tools.answer_with_tools(d["question"], generate_with_logprobs, top_k=3, temperature=0, seed=SEED)
    print("QUESTION :", d["question"], "| gold:", d["gold_answers"][:1])
    print("TOOL ARGS:", out["trace"]["tool_args"], "| called by model:", out["trace"]["tool_called"])
    print("TOP PASSAGE:", (out["retrieved"][0]["title"] + ": " + out["retrieved"][0]["text"][:120] + "...") if out.get("retrieved") else None)
    print("FINAL    :", json.dumps(out["result"]), "\n")

## Numbers to copy into the report (section 10 of `report/Om_Section10_and_Demo_Script.docx`)
* E4 validity first attempt / after retry, most frequent failed rule  -> printed in 4.3
* Confidence: mean when correct vs wrong, ECE -> 4.5
* BM25 hit@1 / hit@3, model-called-tool %, accuracy tool vs paragraph mode -> 5.1 / 5.2
* Limitation: index covers only 20 articles; confidence calibrated only on this 200-question set.